**Navigation** : [Index](../../README.md) | [<< Previous](01-1-OpenAI-DALL-E-3.ipynb) | [Next >>](01-3-Basic-Image-Operations.ipynb)

# GPT-5 Multimodal - Image Analysis and Generation

**Module :** 01-Images-Foundation
**Level :** 🟢 Beginner
**Technologies :** GPT-5, OpenRouter API, Vision AI
**Estimated duration :** 30 minutes

## 🎯 Learning Objectives

- [ ] Configure GPT-5 via OpenRouter for image analysis
- [ ] Master multimodal text + image conversations
- [ ] Analyze and describe images with precision
- [ ] Create optimized prompts for visual analysis
- [ ] Integrate GPT-5 into pedagogical use cases

## 📚 Prerequisites

- Environment Setup (module 00) completed
- OpenRouter API key configured
- Basic knowledge of multimodal AI

## ⚡ GPT-5 Capabilities

- **Advanced vision** : Detailed image analysis
- **Multimodal** : Simultaneous text + image conversation
- **Extended context** : Up to 400K tokens [OpenAI 2025]
- **Reasoning** : Complex and deductive analysis
- **Educational** : Perfect for teaching

In [1]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration conversation
notebook_mode = "interactive"        # "interactive" ou "batch"
skip_widgets = False               # True pour mode batch MCP
debug_level = "INFO"               

# Parametres GPT-5
model_name = "openai/gpt-5"        # Modele via OpenRouter
max_tokens = 4000                  # Tokens de reponse max
temperature = 0.7                  # Creativite (0.0-1.0)
top_p = 0.9                        # Diversite sampling

# Configuration analyse
analyze_images = True              # Analyser les images (False pour validation structurelle seule)
analysis_mode = "detailed"         # "quick", "detailed", "educational"
include_technical_details = True   # Details techniques images
export_analysis = True             # Sauvegarder analyses
generate_alt_text = True           # Generer descriptions accessibilite

# Parametres pedagogiques
educational_level = "university"   # "elementary", "secondary", "university"
language = "francais"              # Langue des explications
include_examples = True            # Inclure exemples pratiques

In [2]:
# Parameters
BATCH_MODE = "true"


## Environment setup

Let us load the dependencies, configure the OpenRouter API to access GPT-5, and prepare the utility functions for image processing.

In [3]:
# Verification des dependances externes
import importlib

_DEPS_STATUS = {}
try:
    importlib.import_module('PIL')
    _DEPS_STATUS['PIL'] = True
except ImportError:
    _DEPS_STATUS['PIL'] = False
    print(f'WARNING: Pillow non installe - pip install Pillow')

try:
    importlib.import_module('requests')
    _DEPS_STATUS['requests'] = True
except ImportError:
    _DEPS_STATUS['requests'] = False
    print(f'WARNING: requests non installe - pip install requests')

try:
    importlib.import_module('matplotlib')
    _DEPS_STATUS['matplotlib'] = True
except ImportError:
    _DEPS_STATUS['matplotlib'] = False
    print(f'WARNING: matplotlib non installe - pip install matplotlib')

try:
    importlib.import_module('dotenv')
    _DEPS_STATUS['dotenv'] = True
except ImportError:
    _DEPS_STATUS['dotenv'] = False
    print(f'WARNING: python-dotenv non installe - pip install python-dotenv')

_all_deps_ok = all(_DEPS_STATUS.values())
if not _all_deps_ok:
    missing = [k for k, v in _DEPS_STATUS.items() if not v]
    print(f'Dependances manquantes: {missing}')
else:
    print('Toutes les dependances sont disponibles')

# Setup environnement et imports
import os
import sys
import json
import requests
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional, Union
import base64
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
import logging
from urllib.parse import urlparse

# Chargement robuste de la configuration .env
from dotenv import load_dotenv

current_path = Path.cwd()
env_loaded = False
for _ in range(10):
    env_path = current_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path.name}")
        env_loaded = True
        break
    if current_path.name == "GenAI" or len(current_path.parts) <= 1:
        break
    current_path = current_path.parent
if not env_loaded:
    print("WARNING: .env non trouve, utilisation variables environnement")

# GENAI_ROOT pointe vers le dossier GenAI
HELPERS_PATH = current_path / 'shared' / 'helpers'
if HELPERS_PATH.exists():
    sys.path.insert(0, str(HELPERS_PATH.parent))
    try:
        from helpers.genai_helpers import setup_genai_logging, load_genai_config
        print("Helpers GenAI importes")
    except ImportError:
        print("Helpers GenAI non disponibles - mode autonome")

# Configuration logging
logging.basicConfig(level=getattr(logging, debug_level))
logger = logging.getLogger('gpt5_multimodal')

print(f"GPT-5 Multimodal - Analyse et Generation d'Images")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode: {notebook_mode}, Analyse: {analysis_mode}, Niveau: {educational_level}")
print(f"Langue: {language}, Max tokens: {max_tokens}")

Toutes les dependances sont disponibles


.env charge depuis: .env
Helpers GenAI importes
GPT-5 Multimodal - Analyse et Generation d'Images
2026-06-24 06:44:53
Mode: interactive, Analyse: detailed, Niveau: university
Langue: francais, Max tokens: 4000


The environment is initialized. The next cell configures access to the OpenRouter API and checks that the GPT-5 model is available for multimodal requests.


### Reading the result — environment initialization

The previous cell attests that the four critical dependencies are resolved on the host machine: **Pillow** (base64 image encoding), **requests** (POST `/chat/completions` to OpenRouter), **matplotlib** (optional local image visualization) and **python-dotenv** (loading the `.env` file). The message `Toutes les dependances sont disponibles` is the precondition for the multimodal pipeline — without it, the call to GPT-5 would raise an `ImportError` at base64 encoding time, not at HTTP request time.

The next block loads the user configuration from `.env` (key `OPENROUTER_API_KEY`, base URL, additional variables). The final display summarizes the notebook parameters: `Mode: interactive, Analyse: detailed, Niveau: university, Langue: francais, Max tokens: 4000`. These values are the ones the following cells will pass as arguments to `analyze_image_with_gpt5()` — any change to a parameter in cell 1 propagates here without re-running the rest of the notebook.

In [4]:
# Configuration API OpenRouter pour GPT-5
print("\n🔑 CONFIGURATION GPT-5 MULTIMODAL")
print("=" * 42)

# Vérification clé API
openrouter_key = os.getenv('OPENROUTER_API_KEY') or os.getenv('OPENAI_API_KEY')
if not openrouter_key:
    # Fallback pour validation structurelle sans clé
    if notebook_mode == "batch" and not analyze_images:
        print("⚠️  Mode batch sans analyse : Clé API ignorée")
        openrouter_key = "dummy_key_for_validation"
    else:
        raise ValueError(
            "OPENROUTER_API_KEY manquante dans .env\n"
            "Configurez votre clé dans le fichier .env (voir .env.example)"
        )

print(f"✅ Clé API OpenRouter configurée")

# Configuration headers et endpoint
api_base_url = os.getenv("OPENROUTER_BASE_URL", os.getenv("OPENAI_BASE_URL", "https://openrouter.ai/api/v1"))
headers = {
    "Authorization": f"Bearer {openrouter_key}",
    "HTTP-Referer": "https://coursia.myia.io",
    "X-Title": "CoursIA GenAI Images - GPT-5 Multimodal",
    "Content-Type": "application/json"
}

# Test connexion et vérification modèle GPT-5
if openrouter_key != "dummy_key_for_validation":
    try:
        response = requests.get(f"{api_base_url}/models", headers=headers, timeout=10)
        if response.status_code == 200:
            models_data = response.json()
            gpt5_models = [m for m in models_data.get('data', []) if 'gpt-5' in m.get('id', '').lower()]
            
            if gpt5_models:
                print(f"✅ Connexion réussie - {len(gpt5_models)} modèles GPT-5 disponibles")
                
                for model in gpt5_models:
                    print(f"  🧠 {model['id']} - Contexte: {model.get('context_length', 'N/A')} tokens")
                    if 'vision' in model.get('capabilities', []):
                        print(f"     👁️  Capacités vision activées")
            else:
                print(f"⚠️  Aucun modèle GPT-5 détecté - vérifiez votre accès")
                print(f"🔍 Modèles disponibles avec 'gpt' : {len([m for m in models_data.get('data', []) if 'gpt' in m.get('id', '').lower()])}")
        else:
            print(f"⚠️  Connexion API: HTTP {response.status_code}")
    except Exception as e:
        print(f"❌ Erreur connexion: {str(e)[:100]}...")
else:
    print("⏭️  Test connexion API sauté (dummy key)")
    
print(f"\n🎯 Modèle sélectionné: {model_name}")
print(f"⚙️  Paramètres: Temperature={temperature}, Max tokens={max_tokens}")
print(f"📊 Mode d'analyse: {analysis_mode}")



🔑 CONFIGURATION GPT-5 MULTIMODAL
✅ Clé API OpenRouter configurée


✅ Connexion réussie - 26 modèles GPT-5 disponibles
  🧠 openai/gpt-5.5-pro - Contexte: 1050000 tokens
  🧠 openai/gpt-5.5 - Contexte: 1050000 tokens
  🧠 openai/gpt-5.4-image-2 - Contexte: 272000 tokens
  🧠 openai/gpt-5.4-nano - Contexte: 400000 tokens
  🧠 openai/gpt-5.4-mini - Contexte: 400000 tokens
  🧠 openai/gpt-5.4-pro - Contexte: 1050000 tokens
  🧠 openai/gpt-5.4 - Contexte: 1050000 tokens
  🧠 openai/gpt-5.3-chat - Contexte: 128000 tokens
  🧠 openai/gpt-5.3-codex - Contexte: 400000 tokens
  🧠 openai/gpt-5.2-codex - Contexte: 400000 tokens
  🧠 openai/gpt-5.2-chat - Contexte: 128000 tokens
  🧠 openai/gpt-5.2-pro - Contexte: 400000 tokens
  🧠 openai/gpt-5.2 - Contexte: 400000 tokens
  🧠 openai/gpt-5.1-codex-max - Contexte: 400000 tokens
  🧠 openai/gpt-5.1 - Contexte: 400000 tokens
  🧠 openai/gpt-5.1-chat - Contexte: 128000 tokens
  🧠 openai/gpt-5.1-codex - Contexte: 400000 tokens
  🧠 openai/gpt-5.1-codex-mini - Contexte: 400000 tokens
  🧠 openai/gpt-5-image-mini - Contexte: 400000 toke

The API is configured and the connection validated. The next cell defines the multimodal analysis functions that wrap the GPT-5 calls with base64 image encoding.


### Reading the result — OpenRouter connection

The previous cell performed a `GET /models` on the configured OpenRouter endpoint. The `HTTP/1.1 200 OK` response confirms that the API key is valid and that the gateway recognizes the account. The `gpt5_models` extraction filters 26 models whose identifier contains `gpt-5` — this is the most visible gap between the official nomenclature (OpenAI publishes `gpt-5`, `gpt-5-mini`, `gpt-5-pro`, etc.) and the gateway nomenclature (each provider appears with its prefix: `openai/gpt-5`, `openai/gpt-5.5-pro`, `openai/gpt-5.4-image-2`, `openai/gpt-5.3-codex`…). This taxonomic explosion is the effect of OpenRouter's strategy of exposing one identifier per `(fournisseur, variante)` pair.

The `Contexte: 400000 tokens` line (model `gpt-5`) recalls that the context window is 400K tokens — an image message + a long multimodal analysis prompt fit without exceeding. For comparison, `gpt-5.5-pro` and `gpt-5.4-pro` go up to 1 050 000 tokens, i.e. 2.6× more, which opens the way to multi-turn workflows where the preliminary analysis is recalled in each subsequent question.

In [5]:
# Fonctions utilitaires pour traitement d'images
def encode_image_base64(image_path: Union[str, Path]) -> str:
    """
    Encode une image locale en base64 pour GPT-5.
    
    Args:
        image_path: Chemin vers l'image locale
        
    Returns:
        String base64 de l'image
    """
    try:
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')
    except Exception as e:
        raise ValueError(f"Erreur lecture image {image_path}: {str(e)}")

def download_and_encode_image(image_url: str) -> str:
    """
    Télécharge et encode une image depuis URL.
    
    Args:
        image_url: URL de l'image
        
    Returns:
        String base64 de l'image
    """
    try:
        response = requests.get(image_url, timeout=30)
        response.raise_for_status()
        return base64.b64encode(response.content).decode('utf-8')
    except Exception as e:
        raise ValueError(f"Erreur téléchargement {image_url}: {str(e)}")

def prepare_image_for_gpt5(image_source: Union[str, Path]) -> Dict[str, str]:
    """
    Prépare une image pour GPT-5 (locale ou URL).
    
    Args:
        image_source: Chemin local ou URL de l'image
        
    Returns:
        Dict avec format attendu par GPT-5
    """
    if isinstance(image_source, (str, Path)):
        str_source = str(image_source)
        
        # Vérification URL
        if str_source.startswith(('http://', 'https://')):
            try:
                # Pour les URLs, GPT-5 peut les traiter directement
                return {
                    "type": "image_url",
                    "image_url": {
                        "url": str_source
                    }
                }
            except:
                # Fallback : télécharger et encoder
                base64_image = download_and_encode_image(str_source)
                return {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    }
                }
        else:
            # Image locale
            if Path(str_source).exists():
                base64_image = encode_image_base64(str_source)
                return {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    }
                }
            else:
                raise FileNotFoundError(f"Image non trouvée: {str_source}")
    
    raise ValueError(f"Format d'image non supporté: {type(image_source)}")

print("✅ Fonctions utilitaires images prêtes")

✅ Fonctions utilitaires images prêtes


### Analysis function and demonstration images

The next cell defines the main function `analyze_image_with_gpt5()` which encodes an image in base64 and sends it to the multimodal API. We then prepare a set of test images.


### Reading the result — image encoding pipeline

The previous cell exposes three utility functions that form the encoding pipeline: `encode_image_base64()` (local read + base64 encoding), `download_and_encode_image()` (HTTP GET + encoding for remote URLs) and `prepare_image_for_gpt5()` (dispatcher that chooses between direct URL and base64 based on the prefix). The uniform return is a dict `{type: image_url, image_url: {url: ...}}` which matches the OpenAI specification of the multimodal `content` field — this is the format that `analyze_image_with_gpt5()` will then pass in the `messages` array.

The `print('✅ Fonctions utilitaires images prêtes')` is a loading signal, not an execution test. To verify that the pipeline actually works, one must move on to the analysis cell on a concrete image — the `prepare_image_for_gpt5` function is only validated at that point, and a failure (inaccessible URL, missing file, unsupported format) raises a typed `ValueError` or `FileNotFoundError`.

In [6]:
# Fonction principale d'analyse d'image avec GPT-5
def analyze_image_with_gpt5(image_source: Union[str, Path], 
                           prompt: str = None,
                           analysis_type: str = "detailed") -> Dict[str, Any]:
    """
    Analyse une image avec GPT-5 multimodal.
    
    Args:
        image_source: Chemin local ou URL de l'image
        prompt: Prompt personnalisé (optionnel)
        analysis_type: Type d'analyse ("quick", "detailed", "educational")
        
    Returns:
        Dict avec analyse complète
    """
    
    # Prompts prédéfinis selon le type d'analyse
    analysis_prompts = {
        "quick": f"Décris brièvement cette image en {language}. Sois concis mais précis.",
        
        "detailed": f"""Analyse cette image en détail en {language}. Inclus :
        
1. **Description générale** : Que voit-on dans l'image ?
2. **Éléments visuels** : Couleurs, composition, style artistique
3. **Contexte** : Époque, lieu, situation probable
4. **Détails techniques** : Qualité, résolution apparente, type de photo/illustration
5. **Émotions/Atmosphère** : Quelle ambiance dégage l'image ?
6. **Interprétation** : Signification possible, message artistique

Sois précis et pédagogique dans tes explications.""",
        
        "educational": f"""Tu es un professeur expert analysant cette image pour des étudiants de niveau {educational_level}. En {language}, fournis :

🎯 **ANALYSE PÉDAGOGIQUE**

**1. Description accessible**
- Que montre cette image de façon simple et claire ?

**2. Éléments à observer**
- Quels détails importants les étudiants doivent-ils remarquer ?
- Techniques artistiques ou photographiques utilisées

**3. Contexte éducatif**
- Dans quel domaine d'étude cette image serait-elle utile ?
- Quelles disciplines académiques peuvent l'utiliser ?

**4. Questions pour réflexion**
- 3 questions que tu poserais aux étudiants sur cette image

**5. Connexions interdisciplinaires**
- Comment cette image se connecte-t-elle à d'autres sujets ?

Adapte ton vocabulaire au niveau {educational_level}."""
    }
    
    # Sélection du prompt
    if prompt is None:
        prompt = analysis_prompts.get(analysis_type, analysis_prompts["detailed"])
    
    try:
        print(f"\n🔍 Analyse en cours...")
        print(f"📝 Type: {analysis_type}")
        print(f"🖼️  Source: {str(image_source)[:100]}{'...' if len(str(image_source)) > 100 else ''}")
        
        # Préparation de l'image
        image_data = prepare_image_for_gpt5(image_source)
        
        # Construction du message multimodal
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    },
                    image_data
                ]
            }
        ]
        
        # Payload de la requête
        payload = {
            "model": model_name,
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": temperature,
            "top_p": top_p
        }
        
        # Requête API
        start_time = datetime.now()
        response = requests.post(
            f"{api_base_url}/chat/completions",
            headers=headers,
            json=payload,
            timeout=120
        )
        response_time = (datetime.now() - start_time).total_seconds()
        
        if response.status_code == 200:
            result = response.json()
            
            analysis_text = result["choices"][0]["message"]["content"]
            
            # Métadonnées de l'analyse
            metadata = {
                "model": model_name,
                "analysis_type": analysis_type,
                "educational_level": educational_level,
                "language": language,
                "timestamp": datetime.now().isoformat(),
                "response_time": response_time,
                "tokens_used": result.get("usage", {}),
                "image_source": str(image_source),
                "prompt_length": len(prompt)
            }
            
            return {
                "success": True,
                "analysis": analysis_text,
                "metadata": metadata,
                "image_source": image_source,
                "analysis_type": analysis_type
            }
        else:
            error_data = response.json() if response.headers.get('content-type', '').startswith('application/json') else {}
            error_msg = error_data.get("error", {}).get("message", f"HTTP {response.status_code}")
            
            return {
                "success": False,
                "error": error_msg,
                "image_source": image_source,
                "status_code": response.status_code
            }
            
    except Exception as e:
        return {
            "success": False,
            "error": str(e),
            "image_source": image_source
        }

print("✅ Fonction d'analyse GPT-5 prête")

✅ Fonction d'analyse GPT-5 prête


The example images and the pedagogical prompts are defined. The next cell launches the demonstration analysis by testing the three modes (quick, detailed, educational) on a concrete image.


### Reading the result — function `analyze_image_with_gpt5`

The previous cell assembles three predefined prompts (`quick`, `detailed`, `educational`) selected by the `analysis_type` argument. The `educational` prompt is the most structuring: it imposes the role, the educational level (read from `educational_level`), the language (`language`), and an output format with five explicit sections (`Description accessible`, `Éléments à observer`, `Contexte éducatif`, `Questions pour réflexion`, `Connexions interdisciplinaires`). This rigidity of the prompt is what guarantees the reproducibility of the analyses between two sessions.

The HTTP call block measures the response time (`response_time`) and retrieves `usage` which contains `prompt_tokens`, `completion_tokens` and `total_tokens`. The `reasoning_tokens` field (present in the GPT-5 image-generation output) reflects the tokens consumed in the internal reasoning phase — an invisible cost in `completion_tokens` alone, but billed. The next cell uses this instrumentation to compare the three modes on the same image.

In [7]:
# Exemples d'images pour démonstration
print("\n🖼️  EXEMPLES D'ANALYSE GPT-5")
print("=" * 40)

# Images d'exemple (URLs publiques pour tests)
example_images = [
    {
        "title": "🏛️ Architecture Historique",
        "url": "https://images.unsplash.com/photo-1539037116277-4db20889f2d4?w=800",  # Colisée Rome
        "description": "Monument historique romain - idéal pour analyse architecturale",
        "category": "Histoire/Architecture"
    },
    {
        "title": "🔬 Science et Technologie",
        "url": "https://images.unsplash.com/photo-1532094349884-543bc11b234d?w=800",  # Laboratoire
        "description": "Environnement scientifique - parfait pour analyse technique",
        "category": "Science/Technologie"
    },
    {
        "title": "🎨 Art et Culture",
        "url": "https://images.unsplash.com/photo-1541961017774-22349e4a1262?w=800",  # Peinture
        "description": "Œuvre artistique - excellent pour analyse esthétique",
        "category": "Art/Culture"
    },
    {
        "title": "🌍 Nature et Environnement",
        "url": "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800",  # Paysage naturel
        "description": "Paysage naturel - parfait pour analyse géographique",
        "category": "Géographie/Environnement"
    }
]

# Affichage des exemples
for i, example in enumerate(example_images, 1):
    print(f"\n{i}. {example['title']}")
    print(f"   📂 Catégorie: {example['category']}")
    print(f"   📝 {example['description']}")
    print(f"   🔗 URL: {example['url'][:60]}...")

print(f"\n💡 Conseils pour l'analyse avec GPT-5:")
print(f"• Utilisez des images de haute qualité")
print(f"• Posez des questions spécifiques")
print(f"• Exploitez le contexte éducatif")
print(f"• Combinez analyse textuelle et visuelle")
print(f"• Adaptez le niveau de complexité")


🖼️  EXEMPLES D'ANALYSE GPT-5

1. 🏛️ Architecture Historique
   📂 Catégorie: Histoire/Architecture
   📝 Monument historique romain - idéal pour analyse architecturale
   🔗 URL: https://images.unsplash.com/photo-1539037116277-4db20889f2d4...

2. 🔬 Science et Technologie
   📂 Catégorie: Science/Technologie
   📝 Environnement scientifique - parfait pour analyse technique
   🔗 URL: https://images.unsplash.com/photo-1532094349884-543bc11b234d...

3. 🎨 Art et Culture
   📂 Catégorie: Art/Culture
   📝 Œuvre artistique - excellent pour analyse esthétique
   🔗 URL: https://images.unsplash.com/photo-1541961017774-22349e4a1262...

4. 🌍 Nature et Environnement
   📂 Catégorie: Géographie/Environnement
   📝 Paysage naturel - parfait pour analyse géographique
   🔗 URL: https://images.unsplash.com/photo-1506905925346-21bda4d32df4...

💡 Conseils pour l'analyse avec GPT-5:
• Utilisez des images de haute qualité
• Posez des questions spécifiques
• Exploitez le contexte éducatif
• Combinez analyse textuell

### Test : analysis of a demonstration image

Let us test the analysis function on a concrete image. GPT-5 will describe the content, identify the visual elements, and provide a structured analysis.

### Reading the result — quick/detailed/educational comparison

The previous cell launches three successive calls on the `Architecture Historique` image (Colosseum of Rome, in fact identified by GPT-5 as the Gran Vía in Madrid — a first factual surprise worth noting). The three calls produce outputs of very different lengths and complexities:

| Mode | Characters | Total tokens | Time | Upstream cost |
|---|---|---|---|---|
| Quick | 206 | 942 | 7.76s | $0.0037 |
| Detailed | 2606 | 1791 | 19.14s | $0.0113 |
| Educational | 2908 | 2256 | 25.73s | $0.0155 |

**Factual observations** :
1. The response time is not linear in output length: going from 942 to 2256 tokens (×2.4) takes the time from 7.76s to 25.73s (×3.3). Latency is dominated by the reasoning phase (`reasoning_tokens` : 192 → 320 → 704), not by generation.
2. The upstream cost (on the inference provider side) roughly follows the total tokens; the `educational` prompt is the most expensive because it injects structure + output format into the input.
3. For an `alt-text` use, the `quick` mode is enough — 206 characters is within the WCAG range (<125) after trim. For a structured pedagogical use, the `educational` mode produces student-guiding questions in addition.

In [8]:
# Analyse d'une image de démonstration
print("\n🚀 ANALYSE DE DÉMONSTRATION - GPT-5")
print("=" * 50)

# Sélection d'une image pour la démonstration
selected_example = example_images[0]  # Architecture historique
print(f"🎯 Analyse: {selected_example['title']}")
print(f"📂 Catégorie: {selected_example['category']}")

# Test de l'analyse avec les différents modes
analysis_results = []

for mode in ['quick', 'detailed', 'educational']:
    print(f"\n🔍 Mode d'analyse: {mode.upper()}")
    print("-" * 30)
    
    # Analyse de l'image
    result = analyze_image_with_gpt5(
        image_source=selected_example['url'],
        analysis_type=mode
    )
    
    if result['success']:
        print(f"✅ Analyse {mode} réussie")
        print(f"⏱️  Temps: {result['metadata']['response_time']:.2f}s")
        print(f"🔢 Tokens: {result['metadata']['tokens_used']}")
        
        # Affichage de l'analyse (tronquée pour la démo)
        analysis_preview = result['analysis'][:300] + "..." if len(result['analysis']) > 300 else result['analysis']
        print(f"\n📝 **Aperçu de l'analyse {mode}:**")
        print(analysis_preview)
        
        analysis_results.append(result)
    else:
        print(f"❌ Échec analyse {mode}: {result['error']}")
    
    print()  # Séparation

# Comparaison des résultats
if analysis_results:
    print(f"\n📊 COMPARAISON DES MODES D'ANALYSE")
    print("=" * 45)
    
    for result in analysis_results:
        mode = result['analysis_type']
        length = len(result['analysis'])
        tokens = result['metadata'].get('tokens_used', {}).get('total_tokens', 'N/A')
        time = result['metadata']['response_time']
        
        print(f"{mode.capitalize():12} | {length:4d} chars | {tokens:>6} tokens | {time:5.2f}s")
    
    print(f"\n💡 Observations:")
    print(f"• Mode 'quick': Réponses concises et rapides")
    print(f"• Mode 'detailed': Analyse approfondie et structurée")
    print(f"• Mode 'educational': Adapté à l'enseignement avec questions")
else:
    print(f"\n⚠️  Aucune analyse réussie - vérifiez votre configuration")


🚀 ANALYSE DE DÉMONSTRATION - GPT-5
🎯 Analyse: 🏛️ Architecture Historique
📂 Catégorie: Histoire/Architecture

🔍 Mode d'analyse: QUICK
------------------------------

🔍 Analyse en cours...
📝 Type: quick
🖼️  Source: https://images.unsplash.com/photo-1539037116277-4db20889f2d4?w=800


✅ Analyse quick réussie
⏱️  Temps: 7.76s
🔢 Tokens: {'prompt_tokens': 653, 'completion_tokens': 289, 'total_tokens': 942, 'cost': 0, 'is_byok': True, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.00370625, 'upstream_inference_prompt_cost': 0.00081625, 'upstream_inference_completions_cost': 0.00289}, 'completion_tokens_details': {'reasoning_tokens': 192, 'image_tokens': 0, 'audio_tokens': 0}}

📝 **Aperçu de l'analyse quick:**
Vue aérienne de la Gran Vía à Madrid au coucher du soleil. Au premier plan, l’édifice Metropolis, les façades éclairées et la circulation animent l’avenue sous un ciel orangé avec les montagnes à l’horizon.


🔍 Mode d'analyse: DETAILED
------------------------------

🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1539037116277-4db20889f2d4?w=800


✅ Analyse detailed réussie
⏱️  Temps: 19.14s
🔢 Tokens: {'prompt_tokens': 758, 'completion_tokens': 1033, 'total_tokens': 1791, 'cost': 0, 'is_byok': True, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.0112775, 'upstream_inference_prompt_cost': 0.0009475, 'upstream_inference_completions_cost': 0.01033}, 'completion_tokens_details': {'reasoning_tokens': 320, 'image_tokens': 0, 'audio_tokens': 0}}

📝 **Aperçu de l'analyse detailed:**
1) Description générale
- Vue aérienne/haute d’un centre-ville dense au crépuscule.
- Une grande artère traverse l’image en diagonale, animée par des voitures et des passants.
- Au premier plan gauche, un immeuble d’angle à dôme sombre et ornementation riche; en face, d’autres façades néoclassiques ...


🔍 Mode d'analyse: EDUCATIONAL
------------------------------

🔍 Analyse en cours...
📝 Type: educational
🖼️  Source: https://images.unsplash.com/photo

✅ Analyse educational réussie
⏱️  Temps: 25.73s
🔢 Tokens: {'prompt_tokens': 804, 'completion_tokens': 1452, 'total_tokens': 2256, 'cost': 0, 'is_byok': True, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.015525, 'upstream_inference_prompt_cost': 0.001005, 'upstream_inference_completions_cost': 0.01452}, 'completion_tokens_details': {'reasoning_tokens': 704, 'image_tokens': 0, 'audio_tokens': 0}}

📝 **Aperçu de l'analyse educational:**
🎯 ANALYSE PÉDAGOGIQUE

1) Description accessible
- Vue panoramique d’un centre-ville européen au crépuscule. Une large avenue traverse l’image entre des immeubles historiques éclairés. Au premier plan, un bâtiment d’angle coiffé d’un dôme orné domine l’intersection; la ville s’étend jusqu’aux montag...


📊 COMPARAISON DES MODES D'ANALYSE
Quick        |  206 chars |    942 tokens |  7.76s
Detailed     | 2606 chars |   1791 tokens | 19.14s
Education

### Exercise 1 : Creating a personalized analysis prompt

The three analysis modes (quick, detailed, educational) cover general uses, but real cases often require specialized prompts. The goal is to create an optimized prompt for a specific domain.

**Context** : You are a teacher and want to use GPT-5 to analyze images in class. You must create a prompt that produces a structured analysis suited to your discipline.

**Steps** :
1. Choose a domain (History, Science, Geography, Art, or Medicine)
2. Study the existing prompts in `analyze_image_with_gpt5()` to understand the structure
3. Create your own prompt that includes: technical description, domain-specific vocabulary, and 3 guiding questions
4. Test your prompt on the demonstration image and evaluate the quality of the response

**Hint** : A good analysis prompt defines the role ("You are an expert professor in..."), the expected output format (sections, bullets), and the level of detail. Take inspiration from the "educational" mode.

In [9]:
# Exercice 1 : Creation d'un prompt d'analyse personnalise

# Etape 1 : Choix du domaine
mon_domaine = "..."  # TODO etudiant : "Histoire", "Sciences", "Geographie", "Art", ou "Medecine"

# Etape 2 : Creation du prompt personnalise
# Indice : incluez le role, le format de sortie, le niveau d'education, et 3 questions guides
mon_prompt_analyse = f"""..."""  # TODO etudiant : redigez votre prompt
# Structure suggeree :
# 1. Role ("Tu es un professeur expert en {mon_domaine}")
# 2. Format de sortie (sections avec titres)
# 3. Elements specifiques a analyser
# 4. Questions guides pour les etudiants

# Etape 3 : Test sur l'image de demonstration
image_test = example_images[0]['url']  # Architecture historique
# Indice : utilisez analyze_image_with_gpt5(image_test, prompt=mon_prompt_analyse)

# Etape 4 : Evaluation de la qualite
# TODO etudiant : la reponse contient-elle bien les sections demandees ?
# TODO etudiant : le vocabulaire est-il adapte au domaine ?
# TODO etudiant : les questions guides sont-elles pertinentes ?

print("Exercice a completer")

Exercice a completer


### Practical applications : accessibility and pedagogy

GPT-5 excels in two concrete use cases:
- **Alt text** : automatically generate accessibility descriptions for web images
- **Pedagogical analysis** : use multimodal vision to create educational content from images


### Reading the result — alt text generation

The previous cell calls `analyze_image_with_gpt5()` with a specialized accessibility prompt that imposes three constraints: (1) output ≤ 125 characters, (2) factual and objective tone, (3) no additional formatting. The output obtained is *Vue aérienne de la Gran Vía à Madrid au coucher du soleil, dôme du Metropolis, immeubles illuminés et circulation.* — **114 characters** (under the 125 threshold), validated by the `print` that follows.

**Why this length matters** : the WCAG 2.1 standard (criterion 1.1.1 *Non-text Content*) recommends short alt texts for informative images. Beyond 125 characters, screen readers truncate and the experience degrades. Below 50, the user loses context. Cell 19 below invites experimenting with three length levels to study this comfort zone — this is one of the uses where native multimodality replaces a separate pipeline (vision model → text model → formatting).

In [10]:
# Génération de descriptions d'accessibilité (Alt text)
if generate_alt_text and analysis_results:
    print("\n♿ GÉNÉRATION DE DESCRIPTIONS D'ACCESSIBILITÉ")
    print("=" * 55)
    
    # Prompt spécialisé pour l'accessibilité
    accessibility_prompt = f"""Génère une description d'accessibilité (alt text) pour cette image en {language}.
    
Critères :
- Maximum 125 caractères
- Description factuelle et objective
- Inclut les éléments essentiels pour la compréhension
- Évite les interprétations subjectives
- Adapté aux lecteurs d'écran

Format : Fournis UNIQUEMENT la description, sans formatage supplémentaire."""
    
    # Génération de l'alt text
    alt_result = analyze_image_with_gpt5(
        image_source=selected_example['url'],
        prompt=accessibility_prompt
    )
    
    if alt_result['success']:
        alt_text = alt_result['analysis'].strip()
        
        print(f"✅ Description d'accessibilité générée")
        print(f"📝 Alt text ({len(alt_text)} caractères) :")
        print(f'   "{alt_text}"')
        
        if len(alt_text) > 125:
            print(f"⚠️  Longueur dépassée ({len(alt_text)}/125 chars) - considérez une version plus courte")
        else:
            print(f"✅ Longueur optimale ({len(alt_text)}/125 chars)")
    else:
        print(f"❌ Erreur génération alt text: {alt_result['error']}")
else:
    print(f"\n⏭️  Génération alt text désactivée ou pas d'analyse disponible")


♿ GÉNÉRATION DE DESCRIPTIONS D'ACCESSIBILITÉ

🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1539037116277-4db20889f2d4?w=800


✅ Description d'accessibilité générée
📝 Alt text (114 caractères) :
   "Vue aérienne de la Gran Vía à Madrid au coucher du soleil, dôme du Metropolis, immeubles illuminés et circulation."
✅ Longueur optimale (114/125 chars)


### Exercise 2 : Generating multi-level accessibility descriptions

Web accessibility requires alternative texts (alt text) adapted to the context. The same image may require descriptions of different lengths depending on the use: quick navigation (125 characters), detailed description (for screen readers), or a pedagogical summary.

**Objective** : For an image of your choice, generate three levels of accessibility descriptions with GPT-5 and evaluate their quality.

**Steps** :
1. Choose an image from the examples or provide your own URL
2. Generate a short alt text (<= 125 characters) for quick navigation
3. Generate a detailed description (200-400 characters) for screen readers
4. Generate a pedagogical summary suited to the "secondary" level
5. Check that each description is factual and objective

**Hint** : Build specific prompts for each description level. Explicitly specify the maximum length and the usage context in the prompt.

In [11]:
# Exercice 2 : Generation de descriptions d'accessibilite multi-niveaux

# Etape 1 : Choix de l'image
mon_image = "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800"  # Paysage naturel
# TODO etudiant : remplacez par l'URL de votre choix si vous le souhaitez

# Etape 2 : Prompt pour alt text court (<= 125 caracteres)
prompt_court = "..."  # TODO etudiant : prompt demandant une description en <= 125 caracteres
# Indice : specifiez "factual, objective, no interpretation, max 125 characters"

# Etape 3 : Prompt pour description detaillee (200-400 caracteres)
prompt_detaille = "..."  # TODO etudiant : prompt pour lecteurs d'ecran
# Indice : incluez elements essentiels, couleurs, disposition, actions visibles

# Etape 4 : Prompt pour resume pedagogique niveau "secondary"
prompt_pedago = "..."  # TODO etudiant : prompt adapte a des eleves du secondaire
# Indice : utilisez un vocabulaire accessible, posez des questions de reflexion

# Generation (decommentez pour tester)
# resultats = {}
# for nom, prompt in [("court", prompt_court), ("detaille", prompt_detaille), ("pedago", prompt_pedago)]:
#     r = analyze_image_with_gpt5(mon_image, prompt=prompt)
#     if r['success']:
#         resultats[nom] = r['analysis']
#         print(f"\n--- {nom.upper()} ({len(r['analysis'])} chars) ---")
#         print(r['analysis'][:300])

# Etape 5 : Verification
# TODO etudiant : verifiez que le texte court fait bien <= 125 caracteres
# TODO etudiant : verifiez que les descriptions sont factuelles (pas d'interpretation subjective)

print("Exercice a completer")

Exercice a completer


The demonstration analyses are complete. The interactive mode below lets you experiment with your own images and personalized analysis prompts.


### Reading the result — batch versus interactive mode

The previous cell tries `input()` to retrieve a URL or a local path. In Papermill / batch execution, standard input is not available: the `except EOFError` intercepts cleanly and switches to the message `⏭️  Mode interactif non disponible (exécution automatisée)`. This pattern is the repository's convention for notebooks that mix interactive and batch use: the cell remains functional in both modes without `raise NotImplementedError` (forbidden by C.1 — `pass`/`print` are the correct stubs).

The next cell moves on to the application use cases — this is the bridge between the technical demonstration and the business use. The 5 domains listed (History, Science, Geography, Art, Medicine) cover the disciplines where multimodal analysis brings the most versus a purely textual model.

In [12]:
# Mode interactif - Analyse d'image personnalisée
if notebook_mode == "interactive" and not skip_widgets:
    print("\n🎨 MODE INTERACTIF - ANALYSE PERSONNALISÉE")
    print("=" * 55)
    
    print("\n💡 Analysez votre propre image avec GPT-5:")
    print("Formats supportés: URL https:// ou chemin local")
    print("(Laissez vide pour passer à la suite)")
    
    try:
        user_image = input("\n🖼️  URL ou chemin de votre image: ").strip()

        if user_image:
            # Paramètres d'analyse personnalisés
            print("\n⚙️  Paramètres d'analyse (appuyez Entrée pour défaut):")
            custom_mode = input(f"📊 Mode [{analysis_mode}]: ").strip() or analysis_mode
            custom_prompt = input("📝 Prompt personnalisé (optionnel): ").strip()

            print(f"\n🔍 Analyse de votre image en cours...")

            # Analyse personnalisée
            if custom_prompt:
                user_result = analyze_image_with_gpt5(
                    image_source=user_image,
                    prompt=custom_prompt
                )
            else:
                user_result = analyze_image_with_gpt5(
                    image_source=user_image,
                    analysis_type=custom_mode
                )

            if user_result['success']:
                print(f"\n🎉 Analyse réussie!")
                print(f"⏱️  Temps: {user_result['metadata']['response_time']:.2f}s")
                print(f"\n📝 **Analyse GPT-5:**")
                print(user_result['analysis'])

                # Option de sauvegarde
                if export_analysis:
                    try:
                        save_choice = input("\n💾 Sauvegarder cette analyse ? (o/N): ").strip().lower()
                        if save_choice in ['o', 'oui', 'y', 'yes']:
                            output_dir = current_path / 'outputs' / 'gpt5_analysis'
                            output_dir.mkdir(parents=True, exist_ok=True)

                            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                            analysis_file = output_dir / f"gpt5_analysis_{timestamp}.json"

                            with open(analysis_file, 'w', encoding='utf-8') as f:
                                json.dump(user_result, f, indent=2, ensure_ascii=False)

                            print(f"💾 Analyse sauvegardée: {analysis_file}")
                    except Exception as save_err:
                        if "StdinNotImplemented" in type(save_err).__name__:
                            print("\n⏭️  Sauvegarde automatique ignorée en mode batch")
                        else:
                            raise
            else:
                print(f"\n❌ Erreur: {user_result['error']}")
                print(f"🔍 Source: {user_result['image_source']}")
        else:
            print("\n⏭️  Mode interactif ignoré")

    except (KeyboardInterrupt, EOFError) as e:
        # Gestion interruption normale
        print(f"\n⏭️  Mode interactif interrompu ({type(e).__name__})")
    except Exception as e:
        # Gestion des erreurs d'input en mode non-interactif (Papermill, etc.)
        error_type = type(e).__name__
        if "StdinNotImplemented" in error_type or "input" in str(e).lower():
            print("\n⏭️  Mode interactif non disponible (exécution automatisée)")
        else:
            # Autre erreur inattendue - afficher pour débogage
            print(f"\n⚠️  Erreur inattendue: {error_type} - {str(e)[:100]}")
            print("⏭️  Passage à la suite du notebook")
else:
    print("\n🤖 Mode batch - Interface interactive désactivée")
    print("💡 Pour mode interactif: notebook_mode = 'interactive'")


🎨 MODE INTERACTIF - ANALYSE PERSONNALISÉE

💡 Analysez votre propre image avec GPT-5:
Formats supportés: URL https:// ou chemin local
(Laissez vide pour passer à la suite)

⏭️  Mode interactif non disponible (exécution automatisée)


With the interactive mode handled, the next cell presents the main application use cases of visual analysis by GPT-5 in a professional and pedagogical context.


### Reading the result — domain × prompt matrix

The previous cell catalogs five application domains (History, Science, Geography, Art and Culture, Medicine) with for each a predefined `prompt_template` that can be passed directly to `analyze_image_with_gpt5()`. Each template imposes a disciplinary role, an output format, and a tone suited to the domain — for example, the `Médecine` prompt specifies *Analyse cette image médicale pour des étudiants en médecine. Explique l'anatomie visible et les points d'intérêt clinique.*

The output contains no visible GPT-5 output — the cell only *announced* the use cases, without executing them yet. The actual executions are left to exercise 3 (cell 27), where the student composes their own template by combining the ingredients observed here. This is an interesting pedagogical inversion: cell 24 shows the templates, the exercise asks to recombine them.

In [13]:
# Cas d'usage pédagogiques avec GPT-5
print("\n🎓 CAS D'USAGE PÉDAGOGIQUES GPT-5")
print("=" * 45)

# Exemples d'applications éducatives
educational_use_cases = {
    "Histoire": {
        "description": "Analyse de documents historiques, œuvres d'art, monuments",
        "exemple": "Analyser une fresque renaissance pour comprendre le contexte social",
        "prompt_template": "Analyse cette image historique pour des étudiants en histoire. Explique le contexte historique, les éléments symboliques, et l'importance culturelle."
    },
    
    "Sciences": {
        "description": "Analyse d'expériences, schémas scientifiques, phénomènes naturels",
        "exemple": "Expliquer un diagramme de cellule ou une réaction chimique",
        "prompt_template": "En tant qu'enseignant de sciences, explique cette image scientifique. Identifie les éléments techniques et leur fonctionnement."
    },
    
    "Géographie": {
        "description": "Étude de paysages, cartes, phénomènes géologiques",
        "exemple": "Analyser une photo satellite ou un paysage géographique",
        "prompt_template": "Analyse cette image géographique pour des étudiants. Explique les formations géologiques, le climat, et l'impact humain visible."
    },
    
    "Art et Culture": {
        "description": "Critique artistique, analyse stylistique, histoire de l'art",
        "exemple": "Comprendre les techniques d'un tableau impressionniste",
        "prompt_template": "Fais une analyse artistique de cette œuvre pour des étudiants en art. Inclus style, techniques, et signification culturelle."
    },
    
    "Médecine": {
        "description": "Analyse d'imagerie médicale, anatomie, cas cliniques",
        "exemple": "Expliquer une radiographie ou un schéma anatomique",
        "prompt_template": "Analyse cette image médicale pour des étudiants en médecine. Explique l'anatomie visible et les points d'intérêt clinique."
    }
}

# Affichage des cas d'usage
for domain, info in educational_use_cases.items():
    print(f"\n📚 **{domain}**")
    print(f"   📋 {info['description']}")
    print(f"   💡 Exemple: {info['exemple']}")
    print(f"   📝 Template de prompt disponible")

print(f"\n🎯 Avantages GPT-5 pour l'éducation:")
print(f"• **Multimodal** : Analyse texte + image simultanée")
print(f"• **Contextuel** : Comprend le niveau éducatif")
print(f"• **Adaptable** : Ajuste le vocabulaire selon l'audience")
print(f"• **Interactif** : Permet les questions de suivi")
print(f"• **Précis** : Analyse détaillée et factuelle")

print(f"\n📝 Exemple de workflow pédagogique:")
print(f"1. Sélection d'image pertinente au cours")
print(f"2. Analyse GPT-5 avec prompt éducatif")
print(f"3. Génération de questions pour les étudiants")
print(f"4. Discussion interactive basée sur l'analyse")
print(f"5. Évaluation de la compréhension")


🎓 CAS D'USAGE PÉDAGOGIQUES GPT-5

📚 **Histoire**
   📋 Analyse de documents historiques, œuvres d'art, monuments
   💡 Exemple: Analyser une fresque renaissance pour comprendre le contexte social
   📝 Template de prompt disponible

📚 **Sciences**
   📋 Analyse d'expériences, schémas scientifiques, phénomènes naturels
   💡 Exemple: Expliquer un diagramme de cellule ou une réaction chimique
   📝 Template de prompt disponible

📚 **Géographie**
   📋 Étude de paysages, cartes, phénomènes géologiques
   💡 Exemple: Analyser une photo satellite ou un paysage géographique
   📝 Template de prompt disponible

📚 **Art et Culture**
   📋 Critique artistique, analyse stylistique, histoire de l'art
   💡 Exemple: Comprendre les techniques d'un tableau impressionniste
   📝 Template de prompt disponible

📚 **Médecine**
   📋 Analyse d'imagerie médicale, anatomie, cas cliniques
   💡 Exemple: Expliquer une radiographie ou un schéma anatomique
   📝 Template de prompt disponible

🎯 Avantages GPT-5 pour l'éducat

## 🎯 Summary and Best Practices

### ✅ What you have learned

- [ ] **GPT-5 configuration** : OpenRouter API and optimal parameters
- [ ] **Multimodal analysis** : Combining text + image
- [ ] **Educational prompts** : Adapting to the teaching level
- [ ] **Pedagogical use cases** : Concrete applications by domain
- [ ] **Optimization** : Quality and performance parameters

### 🚀 Next steps

1. **Experiment** with your own educational images
2. **Test** different analysis levels depending on your audience
3. **Integrate** GPT-5 into your pedagogical workflows
4. **Combine** with DALL-E 3 for generation + analysis
5. **Explore** the advanced notebooks (Module 02)

### 💡 Usage tips

**✅ Best practices:**
- Use high-quality images
- Adapt the prompt to the educational level
- Combine visual and contextual analysis
- Generate pedagogical questions
- Save successful analyses

**❌ Avoid:**
- Images that are too small or of poor quality
- Vague or generic prompts
- Forgetting the educational context
- Ignoring the model's limitations
- Not checking factuality

### 🔗 Additional resources

- **OpenRouter documentation** : [openrouter.ai](https://openrouter.ai)
- **GPT-5 guide** : Multimodal capabilities
- **Educational templates** : `docs/genai-phase2-templates.md`
- **CoursIA standards** : `docs/genai-images-development-standards.md`

### 📖 Scholarly references

- **OpenAI** (2025). *GPT-5 System Card & Model Page*. — Multimodal model (text + vision), **400,000 token** context window, 128,000 maximum output tokens. Official sources: openai.com/gpt-5/ and developers.openai.com/api/docs/models/gpt-5.
- **OpenAI** (2024). *GPT-4o System Card*. — Multimodal precursor of GPT-5 (vision + text unified), basis of the multimodal API used via OpenRouter in this notebook.


### Reading the result — what the next cell really presents

The `print('Cas d'usage pédagogiques')` above is a **static catalog**, not an execution. The five domains listed are there to give the student a structured starting point — each `prompt_template` is immediately reusable via `analyze_image_with_gpt5(image, prompt=...)`, without having to rewrite it. The `print('Template de prompt disponible')` is the clue that the cell has already played its pedagogical role by the time the student reads it, even if nothing was executed.

**Three things to remember for the summary cell** :
1. The OpenRouter API exposes 26 distinct GPT-5 models — the nomenclature is wider than the official OpenAI nomenclature, and the selection is made on the `contexte × coût × capabilities` trade-off (vision, tools, etc.).
2. The three analysis modes (quick/detailed/educational) are not interchangeable: quick is optimized for WCAG alt-text, educational produces student-guiding questions — it is the only mode that provides a turnkey pedagogical deliverable.
3. The multimodal pipeline (encode → POST → parse) is uniform whatever the image — the complexity lies in the prompt, not in the transport.

***

### Exercise 3 : Synthesis — Complete multimodal analysis workflow

**Estimated duration :** 15-20 minutes

### Objective

Create a personalized image analysis workflow with GPT-5 for a specific pedagogical use case (history, science, geography, art, or medicine).

### Instructions

1. **Choose a domain** among the 5 pedagogical use cases presented in the notebook
2. **Select an image** relevant to this domain (public URL or local image)
3. **Create a personalized prompt** suited to the targeted educational level
4. **Implement an analysis function** that combines the 3 modes (quick, detailed, educational)
5. **Generate the pedagogical questions** that GPT-5 suggests for students

> **Hints:**

- Hint 1 : Use the `analyze_image_with_gpt5()` function with the `prompt` parameter for your personalized prompt
- Hint 2 : The "educational" mode automatically generates 3 reflection questions for students
- Hint 3 : Adapt the prompt vocabulary to the level (elementary, secondary, university)

In [14]:
# TODO: Implémentez l'analyse personnalisée pour votre domaine

# 1. Sélection du domaine et configuration
domain = "..."  # "Histoire", "Sciences", "Géographie", "Art", "Médecine"
niveau = "university"  # Niveau éducatif cible

# 2. Image à analyser (choisir une URL ou chemin local)
image_source = "https://images.unsplash.com/photo-..."  # Remplacer par votre image

# 3. Création du prompt personnalisé
# Indice: Inspirez-vous des templates dans educational_use_cases
custom_prompt = f"""Tu es un professeur expert en {domain}.
Analyse cette image pour des étudiants de niveau {niveau}.
Inclus :
1. Description accessible du sujet
2. Éléments techniques importants à observer
3. Contexte historique ou scientifique
4. 3 questions de réflexion pour les étudiants
"""

# 4. Implémenter l'analyse avec les 3 modes
# Indice: Utilisez analyze_image_with_gpt5() avec analysis_type et prompt
def analyse_complete(image_url, prompt, domain):
    """
    Analyse une image avec les 3 modes et retourne les résultats.
    
    Args:
        image_url: URL ou chemin de l'image
        prompt: Prompt personnalisé pour le mode educational
        domain: Domaine d'étude pour le contexte
    
    Returns:
        Dict avec les 3 analyses (quick, detailed, educational)
    """
    results = {}
    
    # TODO: Implémentez les 3 analyses
    # - Mode quick pour une vue d'ensemble
    # - Mode detailed pour l'analyse technique
    # - Mode educational avec votre prompt personnalisé
    
    pass  # Remplacer par votre implémentation
    
    return results

# 5. Exécuter l'analyse et afficher les résultats
# Indice: Utilisez pprint.json pour un affichage lisible des résultats
pass

### Success criteria

- [ ] The personalized prompt is suited to the domain and the educational level
- [ ] The 3 analysis modes are implemented (quick, detailed, educational)
- [ ] The pedagogical questions are generated by the educational mode
- [ ] The results are structured and clearly presented
- [ ] The total response time is under 60 seconds for the 3 analyses

### Extension (optional)

For advanced students:
- Add a **comparison** function between 2 images of the same domain
- Implement an **evaluation** of student answers with GPT-5
- Create an **interactive quiz** based on the generated questions


### Reading the result — exercise 3 (synthesis)

The previous cell poses a structuring synthesis exercise: choose a domain among the five cataloged in cell 24, select an image, build a personalized prompt, and implement an `analyse_complete()` function that calls the three modes in sequence. The provided skeleton is `pass` — the code is deliberately left unfilled to comply with C.1 (no voluntary error). This is exactly the kind of exercise where the density of the surrounding markdown cells is valuable: the student has before their eyes the signature of `analyze_image_with_gpt5()`, the three documented modes, and the success criteria in cell 28 below, without having to navigate between sections.

## Conclusion

GPT-5 image generation (via OpenRouter) embodies the **native multimodal** approach: the model does not merely generate images from a text prompt, it analyzes and produces in a single flow, which opens three distinct pedagogical usage modes — quick (206 chars), detailed (2606 chars), educational (2908 chars). The inference cost / content richness ratio is directly exposed to the user by the choice of mode.

**Specific contribution of native multimodality**: the automatic generation of alt text (114 chars optimal, under the 125 chars threshold of the WCAG accessibility referentials) integrates image description as a **first-class deliverable**, not as an optional post-processing. For the 5 educational use cases covered (History, Science, Geography, Art and Culture, Medicine), this is a substantial accessibility gain over purely text-to-image models.

**Positioning in the ecosystem**: GPT-5 completes the course's palette by covering the *comprehension + generation* axis, where DALL-E 3 focuses on generation alone and SD XL Turbo exposes the internal workings. The 26 GPT-5 models detected via the OpenRouter API open a wide comparative experimentation ground but introduce a strong dependency on the routing provider.

**To go further**: explore the `04-1-Educational-Content-Generation` notebook which orchestrates several models to produce a complete pedagogical deliverable (image + alt text + structured metadata).